# 🔴🟡 Bot Puissance 4 — entraînement type AlphaZero

Le bot apprend **tout seul**, en jouant contre lui-même :

1. **Self-play** : le bot joue des parties contre lui-même. À chaque coup, une recherche MCTS (arbre de Monte-Carlo) guidée par le réseau choisit le coup.
2. **Entraînement** : le réseau apprend à prédire (a) les coups choisis par la recherche (*policy*) et (b) le vainqueur final (*value*).
3. On recommence : un meilleur réseau → une meilleure recherche → de meilleures données.

⚠️ Avant de lancer : **Exécution → Modifier le type d'exécution → GPU (T4)**.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import time, os, random
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)

Device : cuda


## Sauvegarde sur Google Drive
Les checkpoints sont enregistrés sur ton Drive : si Colab se déconnecte, l'entraînement reprend là où il s'était arrêté.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/connect4_az'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    SAVE_DIR = './connect4_az'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Sauvegardes dans :', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sauvegardes dans : /content/drive/MyDrive/connect4_az


## 1. Le jeu
Le plateau est toujours vu **du point de vue du joueur qui doit jouer** : ses pions valent `1`, ceux de l'adversaire `-1`.
Après chaque coup on inverse le plateau. Ça permet d'utiliser un seul réseau pour les deux joueurs.

In [ ]:
ROWS, COLS = 6, 7

def initial_board():
    return np.zeros((ROWS, COLS), dtype=np.int8)

def valid_moves(b):
    return b[0] == 0

def is_win(b, r, c):
    p = b[r, c]
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
        n = 1
        for s in (1, -1):
            rr, cc = r + dr * s, c + dc * s
            while 0 <= rr < ROWS and 0 <= cc < COLS and b[rr, cc] == p:
                n += 1; rr += dr * s; cc += dc * s
        if n >= 4:
            return True
    return False

def step(b, col):
    # Joue 'col' pour le joueur courant (=1).
    # Retourne (plateau vu par le joueur suivant, partie_finie, récompense du joueur qui vient de jouer)
    b = b.copy()
    r = ROWS - 1 - np.count_nonzero(b[:, col])
    b[r, col] = 1
    if is_win(b, r, col):
        return -b, True, 1.0
    if not valid_moves(b).any():
        return -b, True, 0.0
    return -b, False, 0.0

def encode(boards):
    b = np.asarray(boards)
    return np.stack([b == 1, b == -1, np.ones_like(b, dtype=bool)], axis=1).astype(np.float32)

## 2. Le réseau de neurones
Un petit ResNet : en entrée le plateau (3 plans 6×7), en sortie
- **policy** : une probabilité pour chacune des 7 colonnes,
- **value** : un score entre -1 (je perds) et +1 (je gagne).

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return F.relu(x + y)

class Net(nn.Module):
    def __init__(self, ch=64, blocks=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU())
        self.body = nn.Sequential(*[ResBlock(ch) for _ in range(blocks)])
        self.p_head = nn.Sequential(nn.Conv2d(ch, 2, 1, bias=False), nn.BatchNorm2d(2), nn.ReLU(),
                                    nn.Flatten(), nn.Linear(2 * ROWS * COLS, COLS))
        self.v_head = nn.Sequential(nn.Conv2d(ch, 1, 1, bias=False), nn.BatchNorm2d(1), nn.ReLU(),
                                    nn.Flatten(), nn.Linear(ROWS * COLS, 64), nn.ReLU(),
                                    nn.Linear(64, 1), nn.Tanh())
    def forward(self, x):
        h = self.body(self.stem(x))
        return self.p_head(h), self.v_head(h)

@torch.no_grad()
def evaluate(net, boards):
    x = torch.from_numpy(encode(boards)).to(device)
    logits, v = net(x)
    return torch.softmax(logits, 1).cpu().numpy(), v.squeeze(1).cpu().numpy()

print('Paramètres :', sum(p.numel() for p in Net().parameters()))

Paramètres : 449370


## 3. MCTS (recherche arborescente)
La recherche est **batchée** : on fait avancer plusieurs parties en parallèle et on évalue toutes leurs feuilles en un seul passage GPU. C'est ce qui rend l'entraînement faisable sur Colab.

In [ ]:
C_PUCT = 1.5         
DIRICHLET_ALPHA = 1.0  
NOISE_FRAC = 0.25    


class Node:
    __slots__ = ('board', 'valid', 'P', 'N', 'W', 'children', 'expanded', 'terminal', 'value')
    def __init__(self, board, terminal=False, value=0.0):
        # value : valeur d'une position finale pour le joueur au trait (-1 perdu, 0 nul)
        self.board, self.terminal, self.value = board, terminal, value
        self.valid = valid_moves(board)
        self.P = np.zeros(COLS)          # prior du réseau, limité aux coups jouables
        self.N = np.zeros(COLS)          # visites de chaque coup
        self.W = np.zeros(COLS)          # somme des valeurs (vues par le joueur au trait ici)
        self.children, self.expanded = {}, False

    def expand(self, p):
        p = p * self.valid
        s = p.sum()
        self.P = p / s if s > 0 else self.valid / self.valid.sum()
        self.expanded = True

    def select(self):
        # PUCT : valeur moyenne Q + bonus d'exploration proportionnel au prior
        q = np.divide(self.W, self.N, out=np.zeros(COLS), where=self.N > 0)
        u = C_PUCT * self.P * np.sqrt(self.N.sum()) / (1 + self.N)
        return int(np.argmax(np.where(self.valid, q + u, -np.inf)))

    def child(self, a):
        if a not in self.children:
            nb, done, r = step(self.board, a)   # r : récompense du joueur qui vient de jouer
            self.children[a] = Node(nb, done, -r)
        return self.children[a]

def backprop(path, v):
    # v est la valeur vue par le joueur de la feuille
    for node, a in reversed(path):
        v = -v
        node.N[a] += 1
        node.W[a] += v

def run_mcts(net, roots, n_sims, add_noise=False):
    todo = [r for r in roots if not r.expanded]
    if todo:
        p, _ = evaluate(net, [r.board for r in todo])
        for r, pr in zip(todo, p):
            r.expand(pr)
    if add_noise:
        for r in roots:
            noise = np.random.dirichlet([DIRICHLET_ALPHA] * int(r.valid.sum()))
            r.P[r.valid] = (1 - NOISE_FRAC) * r.P[r.valid] + NOISE_FRAC * noise
    for _ in range(n_sims):
        leaves, paths = [], []
        for r in roots:
            node, path = r, []
            while node.expanded and not node.terminal:
                a = node.select()
                path.append((node, a))
                node = node.child(a)
            if node.terminal:
                backprop(path, node.value)
            else:
                leaves.append(node); paths.append(path)
        if leaves:
            p, v = evaluate(net, [l.board for l in leaves])
            for leaf, path, pr, vv in zip(leaves, paths, p, v):
                leaf.expand(pr)
                backprop(path, float(vv))

def bot_move(net, board, n_sims=200):
    net.eval()
    root = Node(board)
    run_mcts(net, [root], n_sims)
    return int(np.argmax(root.N))

## 4. Self-play et entraînement

In [ ]:
def self_play(net, n_games, n_sims, temp_moves):
    net.eval()
    boards = [initial_board() for _ in range(n_games)]
    hist = [[] for _ in range(n_games)]
    active, data, move_no = list(range(n_games)), [], 0
    while active:
        roots = [Node(boards[i]) for i in active]
        run_mcts(net, roots, n_sims, add_noise=True)
        still = []
        for i, root in zip(active, roots):
            pi = root.N / root.N.sum()
            hist[i].append((boards[i], pi))
            # début de partie : on tire au hasard selon pi (exploration), ensuite le meilleur coup
            a = np.random.choice(COLS, p=pi) if move_no < temp_moves else int(np.argmax(root.N))
            nb, done, r = step(boards[i], a)
            if done:
                z = r  # résultat pour le dernier joueur, puis on alterne en remontant
                for b, p in reversed(hist[i]):
                    data.append((b, p, z)); z = -z
            else:
                boards[i] = nb; still.append(i)
        active, move_no = still, move_no + 1
    return data

def train(net, opt, buffer, steps, bs):
    net.train(); tp = tv = 0.0
    bs = min(bs, len(buffer))
    for _ in range(steps):
        b, p, z = zip(*random.sample(buffer, bs))
        x = torch.from_numpy(encode(b)).to(device)
        pi = torch.from_numpy(np.stack(p)).float().to(device)
        z = torch.tensor(z, dtype=torch.float32, device=device)
        logits, v = net(x)
        lp = -(pi * F.log_softmax(logits, 1)).sum(1).mean()
        lv = F.mse_loss(v.squeeze(1), z)
        opt.zero_grad(); (lp + lv).backward(); opt.step()
        tp += lp.item(); tv += lv.item()
    return tp / steps, tv / steps

## 5. Adversaires de test
- **random** : joue au hasard.
- **greedy** : gagne s'il peut, bloque si l'adversaire menace, sinon joue au hasard.

In [ ]:
def random_player(b):
    return int(np.random.choice(np.flatnonzero(valid_moves(b))))

def greedy_player(b):
    valid = np.flatnonzero(valid_moves(b))
    for a in valid:                       # gagner
        _, done, r = step(b, a)
        if done and r == 1: return int(a)
    for a in valid:                       # bloquer
        _, done, r = step(-b, a)
        if done and r == 1: return int(a)
    return int(np.random.choice(valid))

def play_vs(net, opponent, n_games=20, n_sims=50):
    w = d = l = 0
    for g in range(n_games):
        b, bot_turn = initial_board(), (g % 2 == 0)   # on alterne qui commence
        while True:
            a = bot_move(net, b, n_sims) if bot_turn else opponent(b)
            b, done, r = step(b, a)
            if done:
                if r == 0: d += 1
                elif bot_turn: w += 1
                else: l += 1
                break
            bot_turn = not bot_turn
    return w, d, l

### 5 bis. Évaluations contre les bots de l'app
*random* et *greedy* deviennent vite trop faciles. On mesure aussi :
- **vs heuristic** : le `HeuristicBot` de l'app (`puissance4_rules`), porté à l'identique — *medium* (Elo 1200) et *expert* (1600, négamax profondeur 5). C'est lui que le modèle doit battre pour le remplacer.
- **vs snapshot précédent** : progresse-t-on encore ? Ouvertures aléatoires de 4 coups jouées des deux côtés (sinon deux réseaux déterministes rejouent toujours la même partie), avec l'écart d'Elo estimé.
- **sims=0** = le réseau seul, comme le `.tflite` dans l'app ; **sims=50** = avec MCTS.

In [ ]:
import math
import os
import random

import numpy as np
import torch

_COLS, _ROWS = 7, 6
_CENTER_FIRST = (3, 2, 4, 1, 5, 0, 6)
_WIN = 1_000_000
_EXPERT_DEPTH = 5
_WINDOWS = [
    tuple((c + i * dc, r + i * dr) for i in range(4))
    for c in range(_COLS) for r in range(_ROWS)
    for dc, dr in ((1, 0), (0, 1), (1, 1), (1, -1))
    if c + 3 * dc < _COLS and 0 <= r + 3 * dr < _ROWS
]


class _Board:
    """Plateau du HeuristicBot : grid[c][r], rangée 0 en bas, 1 = joueur au
    trait au départ de la recherche, 2 = adversaire ; `turn` = qui joue."""

    def __init__(self, b):
        # b : plateau du notebook (vu par le joueur au trait, rangée 0 en haut)
        self.grid = [[0] * _ROWS for _ in range(_COLS)]
        self.heights = [0] * _COLS
        for c in range(_COLS):
            for r in range(_ROWS):
                v = int(b[_ROWS - 1 - r, c])
                if v:
                    self.grid[c][r] = 1 if v == 1 else 2
                    self.heights[c] = r + 1
        self.turn = 1
        self.history = []

    def valid(self):
        return [c for c in range(_COLS) if self.heights[c] < _ROWS]

    def play(self, c, disc=None):
        self.grid[c][self.heights[c]] = disc or self.turn
        self.heights[c] += 1
        self.history.append(c)
        self.turn = 3 - self.turn

    def undo(self):
        c = self.history.pop()
        self.heights[c] -= 1
        self.grid[c][self.heights[c]] = 0
        self.turn = 3 - self.turn

    def wins_at(self, c, r):
        disc = self.grid[c][r]
        for dc, dr in ((1, 0), (0, 1), (1, 1), (1, -1)):
            n = 1
            for s in (1, -1):
                x, y = c + dc * s, r + dr * s
                while 0 <= x < _COLS and 0 <= y < _ROWS and self.grid[x][y] == disc:
                    n += 1; x += dc * s; y += dr * s
            if n >= 4:
                return True
        return False

    def last_move_won(self):
        c = self.history[-1]
        return self.wins_at(c, self.heights[c] - 1)

    def winning_move_for(self, disc):
        for c in self.valid():
            r = self.heights[c]
            self.grid[c][r] = disc
            wins = self.wins_at(c, r)
            self.grid[c][r] = 0
            if wins:
                return c
        return None


def _negamax(bd, depth, alpha, beta):
    if bd.last_move_won():
        return -_WIN - depth
    if sum(bd.heights) == _COLS * _ROWS:
        return 0
    if depth == 0:
        return _evaluate(bd)
    value = -_WIN * 2
    for c in _CENTER_FIRST:
        if bd.heights[c] < _ROWS:
            bd.play(c)
            score = -_negamax(bd, depth - 1, -beta, -alpha)
            bd.undo()
            value = max(value, score)
            alpha = max(alpha, value)
            if alpha >= beta:
                break
    return value


def _evaluate(bd):
    me, grid = bd.turn, bd.grid
    score = sum(3 if grid[3][r] == me else -3 for r in range(bd.heights[3]))
    for window in _WINDOWS:
        mine = theirs = 0
        for c, r in window:
            d = grid[c][r]
            if d == me: mine += 1
            elif d: theirs += 1
        if theirs == 0: score += (0, 1, 4, 16, 0)[mine]
        if mine == 0: score -= (0, 1, 4, 18, 0)[theirs]
    return score


def heuristic_scores(b):
    """Score négamax (profondeur 5) de chaque colonne, comme
    HeuristicBot.scoreColumns côté Dart."""
    bd, scores = _Board(b), {}
    for c in _CENTER_FIRST:
        if bd.heights[c] < _ROWS:
            bd.play(c)
            scores[c] = -_negamax(bd, _EXPERT_DEPTH - 1, -_WIN * 2, _WIN * 2)
            bd.undo()
    return scores


def heuristic_player(strength, seed=None):
    """Bot de l'app : 'beginner' (800), 'medium' (1200) ou 'expert' (1600)."""
    assert strength in ('beginner', 'medium', 'expert')
    rng = random.Random(seed)

    def play(b):
        bd = _Board(b)
        valid = bd.valid()
        win = bd.winning_move_for(1)
        if strength == 'beginner':
            return win if win is not None and rng.random() < 0.5 else rng.choice(valid)
        forced = win if win is not None else bd.winning_move_for(2)
        if forced is not None:
            return forced
        if strength == 'medium':
            safe = []
            for c in valid:
                bd.play(c)
                if bd.winning_move_for(2) is None:
                    safe.append(c)
                bd.undo()
            cols = safe or valid
            return rng.choices(cols, weights=[4 - abs(c - 3) for c in cols])[0]
        scores = heuristic_scores(b)
        best = max(scores.values())
        return rng.choice([c for c, s in scores.items() if s == best])

    return play


def net_player(net, sims=0):
    """sims=0 : le réseau seul, coup le plus probable (ce que fera le TFLite
    dans l'app). sims>0 : + recherche MCTS (bot_move)."""
    net.eval()
    if sims > 0:
        return lambda b: bot_move(net, b, sims)

    def play(b):
        p, _ = evaluate(net, [b])
        return int(np.argmax(p[0] * valid_moves(b)))

    return play


def random_openings(n, plies, seed=0):
    """Ouvertures aléatoires de `plies` coups sans fin de partie : sans elles,
    deux réseaux en argmax rejouent toujours la même partie."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        b, done = initial_board(), False
        for _ in range(plies):
            b, done, _ = step(b, int(rng.choice(np.flatnonzero(valid_moves(b)))))
            if done:
                break
        if not done:
            out.append(b)
    return out


def match_openings(p1, p2, n_games=20, opening_plies=0, seed=0):
    """(V, N, D) de p1 contre p2. Chaque ouverture est jouée deux fois, une
    fois de chaque côté ; sans ouverture, on alterne qui commence."""
    openings = random_openings((n_games + 1) // 2, opening_plies, seed)
    w = d = l = 0
    for g in range(n_games):
        b = openings[g // 2] if opening_plies else initial_board()
        turn_p1 = g % 2 == 0
        while True:
            a = p1(b) if turn_p1 else p2(b)
            b, done, r = step(b, a)
            if done:
                if r == 0: d += 1
                elif turn_p1: w += 1
                else: l += 1
                break
            turn_p1 = not turn_p1
    return w, d, l


def elo_diff(w, d, l):
    """Écart d'Elo estimé d'après le score (borné vers ±800)."""
    s = min(max((w + 0.5 * d) / max(1, w + d + l), 0.01), 0.99)
    return round(-400 * math.log10(1 / s - 1))


def _load(path):
    n = Net().to(device)
    n.load_state_dict(torch.load(path, map_location=device))
    return n.eval()


def extra_evals(net, prev_path=None, n_games=40, sims=(0, 50),
                strengths=('medium', 'expert')):
    """À appeler dans la boucle d'entraînement, après les évals random/greedy."""
    prev = _load(prev_path) if prev_path and os.path.exists(prev_path) else None
    for s in sims:
        me = net_player(net, s)
        # Ouvertures aussi ici : le réseau (argmax) et l'Expert (négamax) sont
        # quasi déterministes, sans elles 20 parties = 2 parties rejouées.
        res = [f"{st} {match_openings(me, heuristic_player(st, seed=1), n_games, opening_plies=2)}"
               for st in strengths]
        line = f"   sims={s:<3d} vs heuristic " + ' | '.join(res)
        if prev is not None:
            r = match_openings(me, net_player(prev, s), n_games, opening_plies=4)
            line += f" | vs {os.path.basename(prev_path)} {r} ≈ {elo_diff(*r):+d} Elo"
        print(line)


def eval_snapshots(snap_dir, n_games=40, sims=(0, 50)):
    """Évalue chaque snapshot du Drive contre les bots de l'app et contre le
    snapshot précédent. Ne touche pas à l'entraînement : à lancer à la fin,
    ou dans une deuxième session Colab pendant qu'il tourne."""
    snaps = sorted(f for f in os.listdir(snap_dir) if f.endswith('.pt'))
    print('(V/N/D) du snapshot ; vs précédent : ouvertures de 4 coups jouées des deux côtés')
    for i, f in enumerate(snaps):
        print(f)
        extra_evals(_load(os.path.join(snap_dir, f)),
                    os.path.join(snap_dir, snaps[i - 1]) if i else None,
                    n_games, sims)


## 6. Boucle d'entraînement 🚀
Compte environ 2–3 min par itération sur un GPU T4. Après ~10 itérations le bot bat facilement *greedy* ; 40+ itérations donnent un bot solide.
Tu peux relancer cette cellule après une déconnexion : ça reprend depuis le dernier checkpoint.

Toutes les 5 itérations, un **snapshot** des poids est gardé dans `snapshots/` : ce sont eux qui servent à créer les niveaux de difficulté.

In [ ]:
CFG = dict(iterations=100, games_per_iter=128, sims=100, temp_moves=10,
           train_steps=300, batch_size=256, lr=1e-4, buffer_max=150_000,
           eval_every=5, snapshot_every=5)
SNAP_DIR = os.path.join(SAVE_DIR, 'snapshots')
os.makedirs(SNAP_DIR, exist_ok=True)

net = Net().to(device)
opt = torch.optim.Adam(net.parameters(), lr=CFG['lr'], weight_decay=1e-4)
start_it = 0
CKPT = os.path.join(SAVE_DIR, 'checkpoint.pt')
BUFFER_FILE = os.path.join(SAVE_DIR, 'buffer.npz')

def save_buffer(buffer):
    b, p, z = zip(*buffer)
    np.savez(BUFFER_FILE + '.tmp.npz', b=np.stack(b), p=np.stack(p).astype(np.float32),
             z=np.array(z, dtype=np.float32))
    os.replace(BUFFER_FILE + '.tmp.npz', BUFFER_FILE)   # jamais de fichier à moitié écrit

def load_buffer():
    if not os.path.exists(BUFFER_FILE):
        return []
    d = np.load(BUFFER_FILE)
    return list(zip(d['b'], d['p'], d['z'].tolist()))

if os.path.exists(CKPT):
    s = torch.load(CKPT, map_location=device)
    net.load_state_dict(s['net']); opt.load_state_dict(s['opt']); start_it = s['it'] + 1
    for g in opt.param_groups:   # le checkpoint restaure l'ancien lr : on impose celui de CFG
        g['lr'] = CFG['lr']
    print("Reprise à l'itération", start_it + 1)
else:
    torch.save(net.state_dict(), os.path.join(SNAP_DIR, 'net_it000.pt'))   # réseau non entraîné = niveau le plus facile
# Buffer : en reprise, celui encore en mémoire (cellule relancée dans la même
# session), sinon celui du Drive (après une déconnexion) ; vide au premier run.
buffer = (globals().get('buffer') or load_buffer()) if start_it else []
print('Buffer :', len(buffer), 'positions')

for it in range(start_it, CFG['iterations']):
    t0 = time.time()
    data = self_play(net, CFG['games_per_iter'], CFG['sims'], CFG['temp_moves'])
    for b, p, z in data:                        # + symétrie gauche/droite (données x2)
        buffer.append((b, p, z))
        buffer.append((b[:, ::-1].copy(), p[::-1].copy(), z))
    buffer = buffer[-CFG['buffer_max']:]
    t1 = time.time()
    lp, lv = train(net, opt, buffer, CFG['train_steps'], CFG['batch_size'])
    print(f"It {it+1:3d} | +{len(data)} positions | buffer {len(buffer)} | "
          f"loss_pi {lp:.3f} loss_v {lv:.3f} | self-play {t1-t0:.0f}s, train {time.time()-t1:.0f}s")
    save_buffer(buffer)
    torch.save({'net': net.state_dict(), 'opt': opt.state_dict(), 'it': it}, CKPT)
    if (it + 1) % CFG['snapshot_every'] == 0:
        torch.save(net.state_dict(), os.path.join(SNAP_DIR, f'net_it{it+1:03d}.pt'))
        print('   📸 snapshot', f'net_it{it+1:03d}.pt')
    if (it + 1) % CFG['eval_every'] == 0:
        print('   vs random (V/N/D):', play_vs(net, random_player))
        print('   vs greedy (V/N/D):', play_vs(net, greedy_player))
        prev_snap = f"net_it{it + 1 - CFG['eval_every']:03d}.pt"
        extra_evals(net, os.path.join(SNAP_DIR, prev_snap))

It   1 | +2753 positions | buffer 5506 | loss_pi 1.868 loss_v 0.381 | self-play 44s, train 5s
It   2 | +2496 positions | buffer 10498 | loss_pi 1.770 loss_v 0.345 | self-play 38s, train 4s
It   3 | +2668 positions | buffer 15834 | loss_pi 1.706 loss_v 0.338 | self-play 40s, train 4s
It   4 | +2986 positions | buffer 21806 | loss_pi 1.653 loss_v 0.363 | self-play 46s, train 4s
It   5 | +3109 positions | buffer 28024 | loss_pi 1.595 loss_v 0.355 | self-play 46s, train 4s
   📸 snapshot net_it005.pt
   vs random (V/N/D): (20, 0, 0)
   vs greedy (V/N/D): (16, 1, 3)
It   6 | +2945 positions | buffer 33914 | loss_pi 1.556 loss_v 0.358 | self-play 45s, train 4s
It   7 | +3438 positions | buffer 40790 | loss_pi 1.516 loss_v 0.360 | self-play 54s, train 4s
It   8 | +3117 positions | buffer 47024 | loss_pi 1.489 loss_v 0.362 | self-play 48s, train 4s
It   9 | +3249 positions | buffer 53522 | loss_pi 1.473 loss_v 0.383 | self-play 54s, train 4s
It  10 | +3169 positions | buffer 59860 | loss_pi 1.4

### 6 bis. Évaluer tous les snapshots
Indépendant de la boucle : à lancer à la fin, ou dans une **deuxième session Colab** (Drive monté, cellules 1 à 5 bis exécutées) pendant que l'entraînement tourne. Relancer la cellule d'entraînement reprend au checkpoint mais **vide le buffer**.

In [ ]:
eval_snapshots(os.path.join(SAVE_DIR, 'snapshots'))

## 7. Niveaux de difficulté 🎚️
Un niveau = **un snapshot** (réseau plus ou moins entraîné) + **un nombre de simulations MCTS** (réflexion) + **une température** (0 = joue toujours le meilleur coup, 1 = tire au hasard selon les probabilités du réseau, donc fait des erreurs).

`sims=0` veut dire que le bot joue directement avec la sortie du réseau, sans recherche : c'est ce que fera le modèle TFLite seul dans ton appli.
Adapte les noms de snapshots à ceux que tu as (liste affichée ci-dessous).

In [ ]:
SNAP_DIR = os.path.join(SAVE_DIR, 'snapshots')
snaps = sorted(f for f in os.listdir(SNAP_DIR) if f.endswith('.pt'))
print('Snapshots disponibles :', snaps)

def snap_path(name):
    return os.path.join(SNAP_DIR, snaps[-1] if name == 'final' else name)

LEVELS = {
    'tres_facile': dict(snapshot='net_it000.pt', sims=0,   temperature=1.0),
    'facile':      dict(snapshot='net_it005.pt', sims=0,   temperature=1.0),
    'moyen':       dict(snapshot='net_it015.pt', sims=0,   temperature=0.0),
    'difficile':   dict(snapshot='final',        sims=100, temperature=0.0),
    'expert':      dict(snapshot='final',        sims=800, temperature=0.0),
}

_cache = {}
def load_net(path):
    if path not in _cache:
        n = Net().to(device)
        n.load_state_dict(torch.load(path, map_location=device))
        n.eval(); _cache[path] = n
    return _cache[path]

def choose_move(net, board, sims=0, temperature=0.0):
    net.eval()
    valid = valid_moves(board)
    if sims > 0:
        root = Node(board); run_mcts(net, [root], sims)
        scores = root.N.astype(np.float64)
    else:
        p, _ = evaluate(net, [board]); scores = p[0].astype(np.float64)
    scores = scores * valid
    if temperature == 0:
        return int(np.argmax(scores))
    p = scores ** (1.0 / temperature); p /= p.sum()
    return int(np.random.choice(COLS, p=p))

def level_player(level):
    cfg = LEVELS[level]; net = load_net(snap_path(cfg['snapshot']))
    return lambda b: choose_move(net, b, cfg['sims'], cfg['temperature'])

### Calibrer les niveaux
Chaque niveau joue contre *random* et *greedy* (V/N/D). Tu dois voir une progression nette ; sinon, change les snapshots ou les `sims`.

In [ ]:
def match(p1, p2, n_games=20):
    w = d = l = 0
    for g in range(n_games):
        b, first = initial_board(), (g % 2 == 0)
        turn_p1 = first
        while True:
            a = p1(b) if turn_p1 else p2(b)
            b, done, r = step(b, a)
            if done:
                if r == 0: d += 1
                elif turn_p1: w += 1
                else: l += 1
                break
            turn_p1 = not turn_p1
    return w, d, l

for lvl in LEVELS:
    pl = level_player(lvl)
    print(f'{lvl:12s} vs random {match(pl, random_player)} | vs greedy {match(pl, greedy_player)}')

## 8. Joue contre ton bot 🎮

In [ ]:
def show(abs_board):
    sym = {1: '🔴', -1: '🟡', 0: '⚪'}
    for row in abs_board:
        print(' '.join(sym[int(x)] for x in row))
    print(' '.join(f'{i}️⃣' for i in range(1, 8)))

def play_human(level='difficile', human_first=True):
    bot = level_player(level)
    b, turn = initial_board(), 1               # turn : 1 = 🔴, -1 = 🟡
    human = 1 if human_first else -1
    while True:
        show(b * turn)
        if turn == human:
            try:
                a = int(input('Ta colonne (1-7) : ')) - 1
            except ValueError:
                continue
            if not (0 <= a < COLS) or not valid_moves(b)[a]:
                print('Coup invalide.'); continue
        else:
            a = bot(b)
            print(f'🤖 Le bot joue la colonne {a + 1}')
        b, done, r = step(b, a)
        if done:
            show(b * -turn)
            if r == 0: print('Match nul !')
            else: print('Tu as gagné 🎉' if turn == human else 'Le bot a gagné 🤖')
            return
        turn = -turn

play_human(level='difficile', human_first=True)

## 9. Export TensorFlow (SavedModel) et TFLite 📦
Le réseau est reconstruit en Keras et les poids PyTorch y sont copiés.
Deux ajouts pour simplifier l'utilisation dans une appli :
- **Entrée simple** : directement le plateau `(batch, 6, 7)` avec `1 / -1 / 0` ; la conversion en 3 plans est faite dans le modèle.
- **Sortie prête** : `policy` est déjà une probabilité (softmax) et les colonnes pleines valent 0.

In [ ]:
import tensorflow as tf, json, shutil
from tensorflow import keras
L = keras.layers
print('TensorFlow', tf.__version__)

class Encode(L.Layer):
    def call(self, b):                                    # (B,6,7) -> (B,6,7,3)
        me  = tf.cast(tf.equal(b, 1.0), tf.float32)
        opp = tf.cast(tf.equal(b, -1.0), tf.float32)
        return tf.stack([me, opp, tf.ones_like(b)], axis=-1)

class MaskedSoftmax(L.Layer):
    def call(self, inputs):
        logits, b = inputs
        valid = tf.cast(tf.equal(b[:, 0, :], 0.0), tf.float32)   # colonne jouable si la case du haut est vide
        return tf.nn.softmax(logits + (1.0 - valid) * -1e9, axis=-1)

def build_keras(ch=64, blocks=6):
    board = keras.Input(shape=(ROWS, COLS), name='board', dtype='float32')
    def conv_bn_relu(x, filters, k, name, relu=True):
        x = L.Conv2D(filters, k, padding='same', use_bias=False, name=name + '_conv')(x)
        x = L.BatchNormalization(epsilon=1e-5, name=name + '_bn')(x)
        return L.ReLU()(x) if relu else x
    x = conv_bn_relu(Encode(name='encode')(board), ch, 3, 'stem')
    for i in range(blocks):
        y = conv_bn_relu(x, ch, 3, f'b{i}_1')
        y = conv_bn_relu(y, ch, 3, f'b{i}_2', relu=False)
        x = L.ReLU()(L.Add()([x, y]))
    p = L.Flatten()(conv_bn_relu(x, 2, 1, 'p'))
    p = L.Dense(COLS, name='p_fc')(p)
    policy = MaskedSoftmax(name='policy')([p, board])
    v = L.Flatten()(conv_bn_relu(x, 1, 1, 'v'))
    v = L.Dense(64, activation='relu', name='v_fc1')(v)
    value = L.Dense(1, activation='tanh', name='value')(v)
    return keras.Model(board, {'policy': policy, 'value': value})

def copy_weights(km, sd, blocks=6):
    sd = {k: v.detach().cpu() for k, v in sd.items()}
    conv = lambda k: [sd[k].permute(2, 3, 1, 0).numpy()]            # (out,in,kh,kw) -> (kh,kw,in,out)
    bn = lambda k: [sd[f'{k}.{n}'].numpy() for n in ('weight', 'bias', 'running_mean', 'running_var')]
    def fc_flat(k, C):                                                # Flatten NCHW (PyTorch) -> NHWC (Keras)
        w = sd[k + '.weight']; out = w.shape[0]
        return [w.reshape(out, C, ROWS, COLS).permute(2, 3, 1, 0).reshape(-1, out).numpy(), sd[k + '.bias'].numpy()]
    fc = lambda k: [sd[k + '.weight'].T.numpy(), sd[k + '.bias'].numpy()]
    g = km.get_layer
    g('stem_conv').set_weights(conv('stem.0.weight')); g('stem_bn').set_weights(bn('stem.1'))
    for i in range(blocks):
        g(f'b{i}_1_conv').set_weights(conv(f'body.{i}.c1.weight')); g(f'b{i}_1_bn').set_weights(bn(f'body.{i}.b1'))
        g(f'b{i}_2_conv').set_weights(conv(f'body.{i}.c2.weight')); g(f'b{i}_2_bn').set_weights(bn(f'body.{i}.b2'))
    g('p_conv').set_weights(conv('p_head.0.weight')); g('p_bn').set_weights(bn('p_head.1'))
    g('p_fc').set_weights(fc_flat('p_head.4', 2))
    g('v_conv').set_weights(conv('v_head.0.weight')); g('v_bn').set_weights(bn('v_head.1'))
    g('v_fc1').set_weights(fc_flat('v_head.4', 1))
    g('value').set_weights(fc('v_head.6'))

def tflite_predict(path, board):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    it.set_tensor(it.get_input_details()[0]['index'], board[None].astype(np.float32)); it.invoke()
    out = {}
    for d in it.get_output_details():
        out['policy' if d['shape'][-1] == COLS else 'value'] = it.get_tensor(d['index'])[0]
    return out['policy'], out['value']

EXPORT_DIR = os.path.join(SAVE_DIR, 'export')
os.makedirs(EXPORT_DIR, exist_ok=True)
QUANTIZE = False     # True = fichier .tflite ~4x plus petit, précision légèrement réduite

def export_snapshot(snap_file):
    base = 'connect4_' + snap_file.replace('net_', '').replace('.pt', '')
    km = build_keras()
    copy_weights(km, torch.load(os.path.join(SNAP_DIR, snap_file), map_location='cpu'))
    sm_dir = os.path.join(EXPORT_DIR, base + '_savedmodel')
    shutil.rmtree(sm_dir, ignore_errors=True)
    if hasattr(km, 'export'): km.export(sm_dir)
    else: tf.saved_model.save(km, sm_dir)
    conv = tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    if QUANTIZE: conv.optimizations = [tf.lite.Optimize.DEFAULT]
    tfl_path = os.path.join(EXPORT_DIR, base + '.tflite')
    open(tfl_path, 'wb').write(conv.convert())
    return km, sm_dir, tfl_path

# export des snapshots utilisés par les niveaux + fichier de config pour l'appli
used = sorted({snaps[-1] if c['snapshot'] == 'final' else c['snapshot'] for c in LEVELS.values()})
files = {}
for sf in used:
    km, sm_dir, tfl = export_snapshot(sf)
    files[sf] = (km, os.path.basename(sm_dir), os.path.basename(tfl))
    print('✅', sf, '->', files[sf][1], '+', files[sf][2], f'({os.path.getsize(tfl)/1e6:.1f} Mo)')

levels_json = {lvl: dict(tflite=files[snaps[-1] if c['snapshot'] == 'final' else c['snapshot']][2],
                         savedmodel=files[snaps[-1] if c['snapshot'] == 'final' else c['snapshot']][1],
                         sims=c['sims'], temperature=c['temperature'])
               for lvl, c in LEVELS.items()}
json.dump(levels_json, open(os.path.join(EXPORT_DIR, 'levels.json'), 'w'), indent=2)
print(json.dumps(levels_json, indent=2))

## 10. Vérification : PyTorch = Keras = TFLite ✔️
On compare les sorties des trois versions sur des positions aléatoires. Les écarts doivent être minuscules (< 1e-4, un peu plus si `QUANTIZE=True`).

In [ ]:
def random_positions(n):
    out = []
    while len(out) < n:
        b, done = initial_board(), False
        for _ in range(np.random.randint(0, 30)):
            b, done, _ = step(b, random_player(b))
            if done: break
        if not done: out.append(b)
    return out

pos = random_positions(50)
for sf, (km, _, tfl_name) in files.items():
    tnet = load_net(os.path.join(SNAP_DIR, sf))
    p_t, v_t = evaluate(tnet, pos)
    valid = np.stack([valid_moves(b) for b in pos])
    p_t = p_t * valid; p_t /= p_t.sum(1, keepdims=True)
    k = km(np.stack(pos).astype(np.float32), training=False)
    p_k, v_k = np.asarray(k['policy']), np.asarray(k['value'])[:, 0]
    tl = [tflite_predict(os.path.join(EXPORT_DIR, tfl_name), b) for b in pos]
    p_l, v_l = np.stack([x[0] for x in tl]), np.array([x[1][0] for x in tl])
    print(f'{sf}: keras Δpolicy={np.abs(p_t-p_k).max():.2e} Δvalue={np.abs(v_t-v_k).max():.2e} | '
          f'tflite Δpolicy={np.abs(p_t-p_l).max():.2e} Δvalue={np.abs(v_t-v_l).max():.2e}')

# exemple lisible
b = initial_board()
p, v = tflite_predict(os.path.join(EXPORT_DIR, files[used[-1]][2]), b)
print('\nPlateau vide -> policy :', np.round(p, 3), '| value :', float(v[0]))

In [ ]:
# Tout télécharger d'un coup
torch.save(net.state_dict(), os.path.join(EXPORT_DIR, 'connect4_final.pt'))
shutil.make_archive('/content/connect4_export', 'zip', EXPORT_DIR)
try:
    from google.colab import files as colab_files
    colab_files.download('/content/connect4_export.zip')
except ImportError:
    pass